
# Miniprojeto 1 — Pré-Processamento de Dados

**Disciplina:** Sistemas Inteligentes — CIn/UFPE

**Dataset Bruto:** `base_churn_5000_VF2_aluno.csv`

## 1. Preparação

### 1.1. Importação das bibliotecas

In [57]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from IPython.display import display

sns.set_theme(style="whitegrid", palette="deep")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Bibliotecas importadas com sucesso!")


Bibliotecas importadas com sucesso!


### 1.2. Carregamento da base de dados bruta

In [58]:
DATA_BASE = pd.Timestamp("2026-09-21")
FIM_JANELA = DATA_BASE + pd.Timedelta(days=60)

DATA_PATH = Path("../data/base_churn_5000_VF2_aluno.csv")

if not DATA_PATH.exists():
    DATA_PATH = Path("data/base_churn_5000_VF2_aluno.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, sep=None, engine="python")

print(f"Registros: {df.shape[0]}")
print(f"Atributos: {df.shape[1]}")
print("\nPrimeiros registros:")
display(df.head(5))

Registros: 5020
Atributos: 10

Primeiros registros:


,id_cliente,data_aquisicao_cliente,mensalidade_rs,data_chamado,atrasos_6m,tipo_contrato,forma_pagamento,fibra_optica,debito_automatico,data_ocorrencia_churn
0,C0001,21/04/2024,"67,96","09/09/2026, 15/08/2026",0.0,Bienal,Boleto,0,0,NaN
1,C0002,21/02/2025,"65,95",NaN,0.0,Mensal,Cartão,0,0,NaN
2,C0003,21/03/2025,"100,88",28/08/2026,1.0,Anual,Boleto,1,1,NaN
3,C0004,21/03/2025,"94,8","06/08/2026, 28/07/2026",2.0,Mensal,Cartão,0,1,NaN
4,C0005,21/02/2022,"53,68",05/07/2026,2.0,Anual,Pix,0,0,NaN


## 2. Remoção de linhas duplicadas


In [59]:

duplicatas_completas = df.duplicated().sum()

ids_repetidos = df.duplicated(
    subset=["id_cliente"]
).sum()

clientes_unicos = df["id_cliente"].nunique()

print ("--- ANTES DA REMOÇÃO ---")
print(f"Total de registros: {len(df)}")
print(f"Clientes distintos: {clientes_unicos}")
print(f"Linhas totalmente duplicadas: {duplicatas_completas}")
print(f"Ocorrências adicionais de IDs repetidos: {ids_repetidos}")

# Remoção das duplicatas
df_nodup = df.drop_duplicates()

duplicatas_completas = df_nodup.duplicated().sum()

ids_repetidos = df_nodup.duplicated(
    subset=["id_cliente"]
).sum()

clientes_unicos = df_nodup["id_cliente"].nunique()

print ("\n--- DEPOIS DA REMOÇÃO ---")
print(f"Total de registros: {len(df_nodup)}")
print(f"Clientes distintos: {clientes_unicos}")
print(f"Linhas totalmente duplicadas: {duplicatas_completas}")
print(f"Ocorrências adicionais de IDs repetidos: {ids_repetidos}")

--- ANTES DA REMOÇÃO ---
Total de registros: 5020
Clientes distintos: 5000
Linhas totalmente duplicadas: 20
Ocorrências adicionais de IDs repetidos: 20

--- DEPOIS DA REMOÇÃO ---
Total de registros: 5000
Clientes distintos: 5000
Linhas totalmente duplicadas: 0
Ocorrências adicionais de IDs repetidos: 0


## 3. Padronização das datas

Convertemos as datas de aquisição e churn para `datetime`, aceitando os formatos `DD/MM/AAAA` e `AAAA-MM-DD`. As datas de chamados são separadas e armazenadas em listas de datas; campos vazios correspondem a listas vazias. A padronização ocorre antes do filtro temporal.


In [60]:
# Padronização das datas na base de dados.
def padronizar_data(valor):
    if pd.isna(valor) or not str(valor).strip():
        return pd.NaT

    # Permite executar novamente a célula com datas já convertidas.
    if isinstance(valor, pd.Timestamp):
        return valor

    for formato in ("%d/%m/%Y", "%Y-%m-%d"):
        data = pd.to_datetime(
            str(valor).strip(), format=formato, errors="coerce"
        )
        if pd.notna(data):
            return data

    raise ValueError(f"Data inválida: {valor!r}")


def padronizar_chamados(valor):
    if isinstance(valor, list):
        return [padronizar_data(data) for data in valor]

    if pd.isna(valor) or not str(valor).strip():
        return []

    return [
        padronizar_data(parte)
        for parte in str(valor).split(",")
        if parte.strip()
    ]

colunas_datas = [
    "data_aquisicao_cliente",
    "data_ocorrencia_churn",
    "data_chamado"
]

print("--- ANTES DA PADRONIZAÇÃO ---")
display(df_nodup[colunas_datas].head())
display(df_nodup[colunas_datas].dtypes)

df_nodup = df_nodup.copy()

for coluna in ["data_aquisicao_cliente", "data_ocorrencia_churn"]:
    df_nodup[coluna] = pd.to_datetime(
        df_nodup[coluna].apply(padronizar_data)
    )

# Cada célula passa a conter uma lista de datas; ausência vira lista vazia.
df_nodup["data_chamado"] = df_nodup["data_chamado"].apply(
    padronizar_chamados
)

print("--- DEPOIS DA PADRONIZAÇÃO ---")
display(df_nodup[colunas_datas].head())
display(df_nodup[colunas_datas].dtypes)


--- ANTES DA PADRONIZAÇÃO ---


,data_aquisicao_cliente,data_ocorrencia_churn,data_chamado
0,21/04/2024,NaN,"09/09/2026, 15/08/2026"
1,21/02/2025,NaN,NaN
2,21/03/2025,NaN,28/08/2026
3,21/03/2025,NaN,"06/08/2026, 28/07/2026"
4,21/02/2022,NaN,05/07/2026


data_aquisicao_cliente    str
data_ocorrencia_churn     str
data_chamado              str
dtype: object

--- DEPOIS DA PADRONIZAÇÃO ---


,data_aquisicao_cliente,data_ocorrencia_churn,data_chamado
0,2024-04-21,NaT,"[2026-09-09 00:00:00, 2026-08-15 00:00:00]"
1,2025-02-21,NaT,[]
2,2025-03-21,NaT,[2026-08-28 00:00:00]
3,2025-03-21,NaT,"[2026-08-06 00:00:00, 2026-07-28 00:00:00]"
4,2022-02-21,NaT,[2026-07-05 00:00:00]


data_aquisicao_cliente    datetime64[us]
data_ocorrencia_churn     datetime64[us]
data_chamado                      object
dtype: object

## 4. Filtro pelo intervalo de tempo

Selecionamos clientes adquiridos até `21/09/2026` e sem cancelamento anterior a essa data-base, incluindo cancelamentos no próprio dia. A coluna `data_ocorrencia_churn` permanece no tipo `datetime`, com `NaT` quando não há cancelamento registrado. A criação do alvo binário será feita posteriormente.


In [61]:
aquisicao = df_nodup["data_aquisicao_cliente"]
saida = df_nodup["data_ocorrencia_churn"]

print("--- ANTES DO FILTRO ---")
print(f"Total de clientes: {len(df_nodup)}")
print(f"Clientes que cancelaram antes da data-base: {int(saida.lt(DATA_BASE).sum())}")
print(f"Clientes que entraram depois da data-base: {int(aquisicao.gt(DATA_BASE).sum())}")
print(f"Maior data de aquisição: {aquisicao.max():%d/%m/%Y}")
print(f"Menor data de aquisição: {aquisicao.min():%d/%m/%Y}")
print(f"Menor data de churn: {saida.min():%d/%m/%Y}")
print(f"Maior data de churn: {saida.max():%d/%m/%Y}")

# Clientes adquiridos até o corte e sem cancelamento anterior à data base.
elegiveis = (
    aquisicao.le(DATA_BASE)
    & (saida.isna() | saida.ge(DATA_BASE))
)

df_intervalo = df_nodup.loc[elegiveis].copy()

print("\n--- DEPOIS DO FILTRO ---")
print(f"Total de clientes: {len(df_intervalo)}")
print(f"Clientes que cancelaram antes da data-base: {int(saida.loc[elegiveis].lt(DATA_BASE).sum())}")
print(f"Clientes que entraram depois da data-base: {int(aquisicao.loc[elegiveis].gt(DATA_BASE).sum())}")
print(f"Clientes removidos pelo filtro: {len(df_nodup) - len(df_intervalo)}")
print(f"Maior data de aquisição: {aquisicao.loc[elegiveis].max():%d/%m/%Y}")
print(f"Menor data de aquisição: {aquisicao.loc[elegiveis].min():%d/%m/%Y}")
print(f"Menor data de churn: {saida.loc[elegiveis].min():%d/%m/%Y}")
print(f"Maior data de churn: {saida.loc[elegiveis].max():%d/%m/%Y}")

--- ANTES DO FILTRO ---
Total de clientes: 5000
Clientes que cancelaram antes da data-base: 10
Clientes que entraram depois da data-base: 0
Maior data de aquisição: 21/08/2026
Menor data de aquisição: 21/09/2020
Menor data de churn: 15/01/2020
Maior data de churn: 20/11/2026

--- DEPOIS DO FILTRO ---
Total de clientes: 4990
Clientes que cancelaram antes da data-base: 0
Clientes que entraram depois da data-base: 0
Clientes removidos pelo filtro: 10
Maior data de aquisição: 21/08/2026
Menor data de aquisição: 21/09/2020
Menor data de churn: 22/09/2026
Maior data de churn: 20/11/2026


## 5. Interpretação das datas

Criamos três atributos a partir das datas padronizadas:

- `churn`: 1 quando há data de cancelamento e 0 quando ela está ausente.
- `chamados_90d`: quantidade de chamados nos 90 dias até a data-base (de `24/06/2026` a `21/09/2026`, inclusive). Chamados anteriores à aquisição ou posteriores à data-base não são contados; uma lista vazia corresponde a zero chamados.
- `tempo_cliente_meses`: meses completos entre a aquisição e a data-base, inclusive para clientes com cancelamento futuro.

As colunas de datas são preservadas em `df_intervalo`.


In [62]:
df_intervalo = df_intervalo.copy()

# Presença de data de churn = 1; ausência (NaT) = 0.
df_intervalo["churn"] = (
    df_intervalo["data_ocorrencia_churn"].notna().astype(int)
)

# Janela de 90 dias, incluindo a data-base e excluindo o limite inferior.
INICIO_CHAMADOS = DATA_BASE - pd.Timedelta(days=90)

df_intervalo["chamados_90d"] = [
    sum(
        pd.notna(data)
        and INICIO_CHAMADOS < data <= DATA_BASE
        and data >= data_aquisicao
        for data in datas
    )
    for data_aquisicao, datas in zip(
        df_intervalo["data_aquisicao_cliente"],
        df_intervalo["data_chamado"]
    )
]

# Meses completos de relacionamento na data-base.
datas_aquisicao = df_intervalo["data_aquisicao_cliente"]

df_intervalo["tempo_cliente_meses"] = (
    (DATA_BASE.year - datas_aquisicao.dt.year) * 12
    + (DATA_BASE.month - datas_aquisicao.dt.month)
    - (DATA_BASE.day < datas_aquisicao.dt.day).astype(int)
).astype(int)

print("--- ATRIBUTOS DERIVADOS DAS DATAS ---")
display(df_intervalo[[
    "id_cliente",
    "data_ocorrencia_churn",
    "churn",
    "data_chamado",
    "chamados_90d",
    "data_aquisicao_cliente",
    "tempo_cliente_meses"
]].head())

print(f"Clientes com churn (=1): {int(df_intervalo['churn'].sum())}")
print(f"Clientes sem churn (=0): {int(df_intervalo['churn'].eq(0).sum())}")


--- ATRIBUTOS DERIVADOS DAS DATAS ---


,id_cliente,data_ocorrencia_churn,churn,data_chamado,chamados_90d,data_aquisicao_cliente,tempo_cliente_meses
0,C0001,NaT,0,"[2026-09-09 00:00:00, 2026-08-15 00:00:00]",2,2024-04-21,29
1,C0002,NaT,0,[],0,2025-02-21,19
2,C0003,NaT,0,[2026-08-28 00:00:00],1,2025-03-21,18
3,C0004,NaT,0,"[2026-08-06 00:00:00, 2026-07-28 00:00:00]",2,2025-03-21,18
4,C0005,NaT,0,[2026-07-05 00:00:00],1,2022-02-21,55


Clientes com churn (=1): 1772
Clientes sem churn (=0): 3218


## 6. Padronização da grafia

Padronizamos `tipo_contrato` e `forma_pagamento` em letras maiúsculas, sem acentos e sem espaços extras. Por exemplo, `Mensal` e `mensal` tornam-se `MENSAL`, enquanto `Cartão` e `cartao` tornam-se `CARTAO`. Os valores ausentes permanecem ausentes, sem imputação nesta etapa.


In [63]:
colunas_grafia = ["tipo_contrato", "forma_pagamento"]

print("--- ANTES DA PADRONIZAÇÃO DA GRAFIA ---")
for coluna in colunas_grafia:
    display(df_intervalo[coluna].value_counts(dropna=False))

df_intervalo = df_intervalo.copy()

for coluna in colunas_grafia:
    df_intervalo[coluna] = (
        df_intervalo[coluna]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.normalize("NFKD")
        .str.replace(r"[\u0300-\u036f]", "", regex=True)
        .str.upper()
    )

print("\n--- DEPOIS DA PADRONIZAÇÃO DA GRAFIA ---")
for coluna in colunas_grafia:
    display(df_intervalo[coluna].value_counts(dropna=False))


--- ANTES DA PADRONIZAÇÃO DA GRAFIA ---


tipo_contrato
Mensal     2711
Anual      1174
Bienal     1000
mensal       40
anual        25
NaN          25
bienal       15
Name: count, dtype: int64

forma_pagamento
Boleto    1938
Cartão    1687
Pix       1255
cartao      50
boleto      25
NaN         20
PIX         15
Name: count, dtype: int64


--- DEPOIS DA PADRONIZAÇÃO DA GRAFIA ---


tipo_contrato
MENSAL    2751
ANUAL     1199
BIENAL    1015
<NA>        25
Name: count, dtype: Int64

forma_pagamento
BOLETO    1963
CARTAO    1737
PIX       1270
<NA>        20
Name: count, dtype: Int64